# Sostituzione di volti e teste in un video

Notebook per Colab (funziona anche su Kaggle). Non contiene logica propria:
si appoggia a `video_face_roster.py` (analizzatore) e `headless_faceswap.py`
(sostituzione), gli stessi script che girano in locale. Le celle eseguono i
comandi con `subprocess` e liste di argomenti, quindi niente problemi di quoting.

Due modalita':
- **headswap** (predefinita): sostituisce volto, capelli, orecchie e cappello
- **faceswap**: sostituisce solo il volto interno

I modelli sono ONNX e non accettano prompt testuali: il prompt viene registrato
nel config come documentazione, ma non influenza il risultato.


In [ ]:
# 1. Dipendenze e codice
# Nessuna logica duplicata: notebook e CLI usano gli stessi due script.
import subprocess, sys
from pathlib import Path

HERE = Path.cwd()
DL = HERE / "dlfolder"

if not DL.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/hacksider/Deep-Live-Cam.git", str(DL)],
                   check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "onnxruntime-gpu", "opencv-python-headless", "insightface",
                "scipy", "scikit-learn", "numpy<2"], check=True)

# il venv del clone non serve su Colab: si usa l'interprete corrente
PY = sys.executable
SCRIPT = str(HERE)

import onnxruntime as ort
print("Provider ONNX disponibili:", ort.get_available_providers())


## 2. I tuoi file

Metti il video e le foto in `/content/`. Su Colab puoi montare Drive da
`Files > Mount Drive`: i file sono allora in `/content/drive/MyDrive/`.


In [ ]:
VIDEO = "/content/video.mp4"
FOTO = ["/content/foto1.jpg", "/content/foto2.jpg"]   # quante vuoi
OUT = Path("/content/risultati")
OUT.mkdir(exist_ok=True, parents=True)

for p in [VIDEO, *FOTO]:
    print(("OK   " if Path(p).exists() else "MANCA"), p)


## 3. Analizzatore: quante persone sono nel video

Campiona il video, raggruppa i volti per identita' e produce:
- `persone_trovate.txt` con l'elenco e il primo frame in cui compare ciascuna
- `persona_N_provino.png`, un campione visivo per ogni persona
- `swap_config.json`, il file di configurazione per lo swap

Il raggruppamento media le embedding su tracciati temporali e sceglie la soglia
con il metodo di Otsu: non dipende da una costante arbitraria.


In [ ]:
CENS = OUT / "censimento"
rc = subprocess.run([PY, f"{SCRIPT}/video_face_roster.py",
                     "--video", VIDEO,
                     "--out", str(CENS),
                     "--mode", "headswap",
                     "--prompt", "sostituisci il volto e i capelli con la foto",
                     "--samples", "90"]).returncode
print("exit code:", rc, "(0 = censimento riuscito)")


## 4. Sostituzione

Dal config: video, foto e modalita' vengono letti da `swap_config.json`.

Se nel video c'e' piu' di una persona, l'abbinamento automatico e' per
frequenza e non puo' sapere a chi appartiene il soggetto delle tue foto.
Controlla `persone_trovate.txt` e i provini, e in quel caso fissalo con
`--map`, dove il numero e' l'indice della foto (`0` = `FOTO[0]`).


In [ ]:
rc = subprocess.run([PY, f"{SCRIPT}/headless_faceswap.py",
                     "--config", str(CENS / "swap_config.json"),
                     "--output", str(OUT / "risultato_headswap.mp4")]).returncode
print("exit code:", rc)
print("codice 3 = nessun volto sostituito, il video non e' adatto")


In [ ]:
# In alternativa tutto in linea, con l'abbinamento fissato a mano.
# --map 0,1 = persona 1 usa FOTO[0], persona 2 usa FOTO[1]
rc = subprocess.run([PY, f"{SCRIPT}/headless_faceswap.py",
                     "--video", VIDEO,
                     "--photo", *FOTO,
                     "--mode", "headswap",
                     "--map", "0,1",
                     "--output", str(OUT / "risultato_headswap.mp4")]).returncode
print("exit code:", rc)


## 5. Verifica

Non dare per scontato che sia riuscito. Il comando esce con codice diverso da
zero se non ha sostituito nessun volto, e `verifica_swap.py` misura quanto e'
residua l'identita' del volto originale: sotto 0.45 il volto e' stato
davvero sostituito.


In [ ]:
rc = subprocess.run([PY, f"{SCRIPT}/verifica_swap.py",
                     "--videoorig", VIDEO,
                     "--videoswap", str(OUT / "risultato_headswap.mp4"),
                     "--rif", FOTO[0],
                     "--modalita", "headswap"]).returncode
print("exit code verifica:", rc, "(0 = coerente)")


In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(OUT / "risultato_headswap.mp4")))
